# Fine-tuning A-ESRGAN para Realce de Fotografía Antigua

Realce/limpieza (deblur + denoise) de fotos antiguas. Dataset domain-matched:

- **HR = "vintage nítido"**: imágenes nítidas (DIV2K) toneadas a sepia/B&N con
  `DamageConfig.tone_only()` (solo color, sin tocar nitidez).
- **LR = degradación realista**: en entrenamiento la genera BasicSR al vuelo
  (Real-ESRGAN, alto orden); para validación reproducible se usa
  `degrade_for_sr` (blur + downsample + ruido + JPEG).

## 0. Setup

In [ ]:
# Celda 1: Dependencias base
%pip install -q \
    kagglehub \
    opencv-python \
    matplotlib \
    numpy \
    tqdm \
    scikit-image \
    pandas \
    lpips \
    omegaconf \
    hydra-core \
    "basicsr>=1.3.3.11" \
    "facexlib>=0.2.0.3" \
    "gfpgan>=0.2.1"

In [ ]:
# Celda 2: Pin de versiones críticas para evitar bug numpy._core durante la sesión
#
# Cuando varios pip installs ocurren durante la sesión (LaMa requirements, basicsr,
# pytorch-msssim, etc.) pueden mezclar archivos de distintas versiones de numpy y
# romper imports con:
#     ImportError: cannot import name '_center' from 'numpy._core.umath'
#
# Aquí forzamos un set internamente consistente con --force-reinstall y dejamos un
# PIP_CONSTRAINT para que los installs posteriores no degraden el pin.

import subprocess, sys, os
from pathlib import Path

PINS = [
    'numpy==2.0.2',           # último parche estable de la rama 2.0 (compatible con scipy 1.14, skimage 0.24)
    'scipy==1.14.1',          # última 1.14 → exige numpy >= 2.0, < 2.2
    'scikit-image==0.24.0',   # última 0.24 → compatible con numpy 2.0
]

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', *PINS],
    check=True,
)

# Constraint file: pip lo respetará en las celdas siguientes (no upgrade implícito)
_CONSTRAINT = Path('/content/numpy_pins.txt')
_CONSTRAINT.write_text('\n'.join(PINS))
os.environ['PIP_CONSTRAINT'] = str(_CONSTRAINT)

print('Pin aplicado:', PINS)
print('PIP_CONSTRAINT activo →', os.environ['PIP_CONSTRAINT'])


In [ ]:
# Celda 3: Imports y configuración global
%matplotlib inline

import os, sys, site, shutil, subprocess
from pathlib import Path
from typing import Optional, Sequence
import cv2
import numpy as np
from PIL import Image, ImageFile
import matplotlib.pyplot as plt
import torch
from IPython.display import display

ImageFile.LOAD_TRUNCATED_IMAGES = True

try:
    from google.colab import output
    IN_COLAB = True
except Exception:
    IN_COLAB = False

WORK_DIR = Path('/content/finetune_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)

AESRGAN_REPO_DIR = Path('/content/A-ESRGAN')

# Directorios de datos
SR_TRAIN_DIR    = WORK_DIR / 'data' / 'aesrgan_train'
SR_VAL_DIR      = WORK_DIR / 'data' / 'aesrgan_val'

# Directorios de LaMa ya provienen de Drive (Fase 2); aqui solo A-ESRGAN.
for d in [SR_TRAIN_DIR/'HR',       SR_TRAIN_DIR/'LR',
          SR_VAL_DIR/'HR',         SR_VAL_DIR/'LR',
          WORK_DIR/'checkpoints'/'aesrgan_finetuned',
          WORK_DIR/'results']:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'GPU disponible: {torch.cuda.is_available()}')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# Google Drive: para persistir los checkpoints del fine-tuning.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/TFM')
DRIVE_CKPT_DIR = PROJECT_DIR / 'checkpoints' / 'aesrgan_finetuned'
DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
print('Checkpoints SR (Drive):', DRIVE_CKPT_DIR)

In [ ]:
# Celda 4: Parche Pillow 11.x — añadir setter a JpegImageFile.mode
# Motivo: el JpegImagePlugin asigna self.mode='RGB' en SOF(), pero en
# Pillow 11.x mode es @property sin setter → AttributeError al abrir JPEGs.
# La extensión C (_imaging.so) no se puede descargar; hay que restaurar 11.3.0 y parchear.
import subprocess, sys

# Restaurar Python-files de Pillow 11.3.0 (deben coincidir con la C-extension en memoria)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'Pillow==11.3.0'],
    check=True
)

# Limpiar caché de módulos PIL (C-extension se reusa desde memoria, Python-files se recargan)
for mod_name in [k for k in list(sys.modules.keys()) if k.lower().startswith('pil')]:
    del sys.modules[mod_name]

# Reimportar PIL desde disco (Python-files 11.3.0 + C-extension 11.3.0 en memoria)
import PIL
from PIL import Image, ImageFile
import PIL.JpegImagePlugin
ImageFile.LOAD_TRUNCATED_IMAGES = True
print(f"Pillow restaurado: {PIL.__version__}")

# Parchear la propiedad mode read-only en JpegImageFile
# En PIL 11.x, mode pasó a ser @property sin setter, pero el plugin JPEG
# aún hace self.mode = "RGB" en el handler SOF → AttributeError
_patched = False
for cls in PIL.JpegImagePlugin.JpegImageFile.__mro__:
    if 'mode' in cls.__dict__:
        prop = cls.__dict__['mode']
        if isinstance(prop, property) and prop.fset is None:
            def _mode_setter(self, value):
                object.__setattr__(self, '_mode', value)
            cls.mode = prop.setter(_mode_setter)
            print(f"✓ Setter añadido a {cls.__name__}.mode")
            _patched = True
        elif isinstance(prop, property):
            print(f"✓ {cls.__name__}.mode ya tiene setter — sin parche")
        else:
            print(f"✓ {cls.__name__}.mode es atributo normal — sin parche")
        break

# Test JPEG round-trip
import io
_t = Image.new('RGB', (16, 16), (200, 100, 50))
_buf = io.BytesIO(); _t.save(_buf, 'JPEG'); _buf.seek(0)
_loaded = Image.open(_buf).convert('RGB')
assert _loaded.size == (16, 16), "JPEG round-trip falló"
print(f"✓ JPEG open/convert OK — size={_loaded.size}, mode={_loaded.mode}")

In [ ]:
# Parches de compatibilidad: basicsr/torchvision (degradations) y PIL para lpips

# Parche basicsr: torchvision >= 0.16 eliminó functional_tensor
for base in site.getsitepackages() + [site.getusersitepackages()]:
    p = Path(base) / 'basicsr' / 'data' / 'degradations.py'
    if p.exists():
        text = p.read_text()
        old  = 'from torchvision.transforms.functional_tensor import rgb_to_grayscale'
        new  = 'from torchvision.transforms.functional import rgb_to_grayscale'
        if old in text:
            p.write_text(text.replace(old, new))
            print(f'Parche aplicado: {p}')

# Parche PIL._typing para lpips/torchvision
import PIL._typing
if not hasattr(PIL._typing, '_Ink'):
    from typing import Union
    PIL._typing._Ink = Union[int, tuple]

print('Parches de compatibilidad aplicados')

## 1. Datos: pares HR/LR

In [ ]:
import sys
from pathlib import Path
import numpy as np
from PIL import Image

PROJECT_DIR = Path('/content/drive/MyDrive/TFM')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from synthetic_degradation import DamageConfig, degrade_for_sr
from synthetic_degradation.global_effects import apply_global

SR_DATA   = PROJECT_DIR / 'data' / 'aesrgan'
HR_DIR    = SR_DATA / 'HR'
VAL_HR    = SR_DATA / 'val' / 'HR'
VAL_LR    = SR_DATA / 'val' / 'LR'
for d in (HR_DIR, VAL_HR, VAL_LR):
    d.mkdir(parents=True, exist_ok=True)
print('Dataset SR (Drive):', SR_DATA)

In [ ]:
# Fuente HR nítida. Ajusta DIV2K_SLUG al dataset DIV2K que uses en Kaggle.
DIV2K_SLUG = 'joe1995/div2k-dataset'   # <-- ajustar si tu slug difiere
MAX_IMAGES = 300                        # acota tiempo/espacio en Colab
MAX_SIDE   = 1024                       # reescala lados grandes para ahorrar

def load_sharp_images(limit=MAX_IMAGES):
    root = None
    try:
        import kagglehub
        root = Path(kagglehub.dataset_download(DIV2K_SLUG))
    except Exception as e:
        print('kagglehub/DIV2K no disponible:', e)
    paths = []
    if root and root.exists():
        for ext in ('*.png', '*.jpg', '*.jpeg'):
            paths.extend(sorted(root.rglob(ext)))
    if not paths:
        print('Sin DIV2K; generando imágenes sintéticas nítidas (humo).')
        imgs, rng = [], np.random.default_rng(0)
        for _ in range(min(limit, 12)):
            imgs.append(rng.integers(0, 256, size=(512, 512, 3), dtype=np.uint8))
        return imgs
    imgs = []
    for p in paths[:limit]:
        im = Image.open(p).convert('RGB')
        if max(im.size) > MAX_SIDE:
            r = MAX_SIDE / max(im.size)
            im = im.resize((int(im.size[0]*r), int(im.size[1]*r)), Image.LANCZOS)
        imgs.append(np.array(im))
    return imgs

sharp_images = load_sharp_images()
print(f'{len(sharp_images)} imágenes nítidas cargadas.')

In [ ]:
SEED = 1234
VAL_FRACTION = 0.1
tone_cfg = DamageConfig.tone_only()

idx = np.arange(len(sharp_images))
np.random.default_rng(SEED).shuffle(idx)
n_val = max(1, round(VAL_FRACTION * len(idx)))
val_ids = set(idx[:n_val].tolist())

n_train = n_valw = 0
for i, img in enumerate(sharp_images):
    rng = np.random.default_rng(SEED + i)
    hr = apply_global(img, rng, tone_cfg)          # nítido + tono de época
    stem = f'img{i:04d}'
    if i in val_ids:
        h, w = hr.shape[:2]; h -= h % 4; w -= w % 4
        hr = hr[:h, :w]
        lr = degrade_for_sr(hr, np.random.default_rng(SEED + 10_000 + i), scale=4)
        Image.fromarray(hr).save(VAL_HR / f'{stem}.png')
        Image.fromarray(lr).save(VAL_LR / f'{stem}.png')
        n_valw += 1
    else:
        Image.fromarray(hr).save(HR_DIR / f'{stem}.png')
        n_train += 1
print(f'HR train: {n_train} | val (LR/HR): {n_valw}')

In [ ]:
# RealESRGANDataset suele requerir un meta_info con los nombres de los HR.
META_INFO = SR_DATA / 'meta_info_HR.txt'
hr_files = sorted(p.name for p in HR_DIR.glob('*.png'))
META_INFO.write_text('\n'.join(hr_files) + '\n')
print(f'meta_info: {META_INFO} ({len(hr_files)} entradas)')

In [ ]:
import matplotlib.pyplot as plt
val_stems = sorted(p.stem for p in VAL_HR.glob('*.png'))[:3]
fig, axes = plt.subplots(len(val_stems), 2, figsize=(8, 4*len(val_stems)))
if len(val_stems) == 1: axes = axes[None, :]
for r, stem in enumerate(val_stems):
    hr = Image.open(VAL_HR / f'{stem}.png')
    lr = Image.open(VAL_LR / f'{stem}.png').resize(hr.size, Image.NEAREST)
    axes[r][0].imshow(lr); axes[r][0].set_title('LR degradada (x4 nearest)'); axes[r][0].axis('off')
    axes[r][1].imshow(hr); axes[r][1].set_title('HR (GT, vintage nítido)'); axes[r][1].axis('off')
plt.tight_layout(); plt.show()
print('Compara la LR con un recorte de foto antigua real: deben parecerse (blandura/grano/tono).')

## 2. Fine-tuning

In [ ]:
# Celda 17: Clonar A-ESRGAN y descargar pesos pre-entrenados
AESRGAN_REPO_URL   = 'https://github.com/stroking-fishes-ml-corp/A-ESRGAN.git'
AESRGAN_MODEL_URL  = 'https://github.com/stroking-fishes-ml-corp/A-ESRGAN/releases/download/v1.0.0/A_ESRGAN_Single.pth'
AESRGAN_PRETRAINED = AESRGAN_REPO_DIR / 'experiments' / 'pretrained_models' / 'A_ESRGAN_Single.pth'

if not AESRGAN_REPO_DIR.exists():
    subprocess.run(['git', 'clone', AESRGAN_REPO_URL, str(AESRGAN_REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(AESRGAN_REPO_DIR), 'pull'], check=True)

AESRGAN_PRETRAINED.parent.mkdir(parents=True, exist_ok=True)
if not AESRGAN_PRETRAINED.exists():
    subprocess.run(['wget', '-q', '-O', str(AESRGAN_PRETRAINED), AESRGAN_MODEL_URL], check=True)
    print('Pesos A-ESRGAN descargados')
else:
    print('Pesos ya disponibles:', AESRGAN_PRETRAINED)

# Listar scripts de entrenamiento disponibles
train_scripts = sorted(AESRGAN_REPO_DIR.rglob('train*.py'))
print('Scripts de entrenamiento encontrados:')
for s in train_scripts:
    print(f'  {s.relative_to(AESRGAN_REPO_DIR)}')

In [ ]:
# NOTA: si el repo A-ESRGAN usa otro network_d (discriminador U-Net de atención propio) o param_key_g, ajustar esos campos al leer sus configs en Colab.
yaml_text = f"""
name: finetune_AESRGAN_vintage
model_type: RealESRGANModel
scale: 4
num_gpu: 1
manual_seed: 0

# --- degradación de alto orden (afinada hacia foto antigua) ---
gt_usm: true
l1_gt_usm: true
percep_gt_usm: true
gan_gt_usm: false
resize_prob: [0.2, 0.7, 0.1]
resize_range: [0.15, 1.5]
gaussian_noise_prob: 0.6
noise_range: [1, 30]
poisson_scale_range: [0.05, 3.0]
gray_noise_prob: 0.5
jpeg_range: [30, 90]
second_blur_prob: 0.8
resize_prob2: [0.3, 0.4, 0.3]
resize_range2: [0.3, 1.2]
gaussian_noise_prob2: 0.6
noise_range2: [1, 25]
poisson_scale_range2: [0.05, 2.5]
gray_noise_prob2: 0.5
jpeg_range2: [30, 90]
gt_size: 128
queue_size: 160

datasets:
  train:
    name: vintage_hr
    type: RealESRGANDataset
    dataroot_gt: {HR_DIR}
    meta_info: {META_INFO}
    io_backend: {{type: disk}}
    blur_kernel_size: 21
    kernel_list: ['iso','aniso','generalized_iso','generalized_aniso','plateau_iso','plateau_aniso']
    kernel_prob: [0.45, 0.25, 0.12, 0.03, 0.12, 0.03]
    sinc_prob: 0.1
    blur_sigma: [0.2, 3.0]
    betag_range: [0.5, 4.0]
    betap_range: [1, 2]
    blur_kernel_size2: 21
    kernel_list2: ['iso','aniso','generalized_iso','generalized_aniso','plateau_iso','plateau_aniso']
    kernel_prob2: [0.45, 0.25, 0.12, 0.03, 0.12, 0.03]
    sinc_prob2: 0.1
    blur_sigma2: [0.2, 1.5]
    betag_range2: [0.5, 4.0]
    betap_range2: [1, 2]
    final_sinc_prob: 0.8
    gt_size: 128
    use_hflip: true
    use_rot: false
    use_shuffle: true
    num_worker_per_gpu: 2
    batch_size_per_gpu: 2
    dataset_enlarge_ratio: 1
    prefetch_mode: ~

network_g:
  type: RRDBNet
  num_in_ch: 3
  num_out_ch: 3
  num_feat: 64
  num_block: 23
  num_grow_ch: 32

network_d:
  type: UNetDiscriminatorSN
  num_in_ch: 3
  num_feat: 64
  skip_connection: true

path:
  pretrain_network_g: /content/A-ESRGAN/experiments/pretrained_models/A_ESRGAN_Single.pth
  param_key_g: params_ema
  strict_load_g: false
  resume_state: ~

train:
  ema_decay: 0.999
  optim_g: {{type: Adam, lr: !!float 1e-4, weight_decay: 0, betas: [0.9, 0.99]}}
  optim_d: {{type: Adam, lr: !!float 1e-4, weight_decay: 0, betas: [0.9, 0.99]}}
  scheduler: {{type: MultiStepLR, milestones: [1500], gamma: 0.5}}
  total_iter: 2000
  warmup_iter: -1
  pixel_opt: {{type: L1Loss, loss_weight: 1.0, reduction: mean}}
  perceptual_opt:
    type: PerceptualLoss
    layer_weights: {{'conv1_2': 0.1, 'conv2_2': 0.1, 'conv3_4': 1.0, 'conv4_4': 1.0, 'conv5_4': 1.0}}
    vgg_type: vgg19
    use_input_norm: true
    range_norm: false
    perceptual_weight: 1.0
    style_weight: 0
    criterion: l1
  gan_opt: {{type: GANLoss, gan_type: vanilla, real_label_val: 1.0, fake_label_val: 0.0, loss_weight: 0.1}}
  net_d_iters: 1
  net_d_init_iters: 0

logger:
  print_freq: 100
  save_checkpoint_freq: 500
  use_tb_logger: false

dist_params:
  backend: nccl
  port: 29500
"""
CONFIG_PATH = Path('/content/finetune_work/aesrgan_finetune.yml')
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
CONFIG_PATH.write_text(yaml_text)
print('Config guardada:', CONFIG_PATH)
print('model_type=RealESRGANModel | degradación al vuelo | gt_size=128')

In [ ]:
# Celda 19: Fine-tuning A-ESRGAN (lanza basicsr/train.py como subprocess)
# Antes del subprocess se libera VRAM del kernel principal (modelo LaMa
# y caché PyTorch) porque BasicSR carga generador+discriminador+VGG19 y
# necesita ~3-4 GB libres para entrenar con batch=2, gt=128.

import subprocess, sys, os, gc, torch

# ─── PASO 1: Liberar VRAM del kernel principal (LaMa + caché PyTorch) ───────
print("=== Liberando VRAM del kernel ===")
print(f"  Antes  → alloc: {torch.cuda.memory_allocated()/1e9:.2f} GB  "
      f"reserved: {torch.cuda.memory_reserved()/1e9:.2f} GB  "
      f"libre: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

# Eliminar tensores y modelos GPU del namespace global
_drop = []
for _name, _obj in list(globals().items()):
    if _name.startswith('_'):
        continue
    if isinstance(_obj, torch.Tensor) and _obj.is_cuda:
        _drop.append(_name)
    elif isinstance(_obj, torch.nn.Module):
        try: _obj.cpu()
        except: pass
        _drop.append(_name)

for _name in _drop:
    try: del globals()[_name]
    except: pass
    print(f"  del {_name}")

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"  Después → alloc: {torch.cuda.memory_allocated()/1e9:.2f} GB  "
      f"libre: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

# ─── PASO 2: Lanzar entrenamiento A-ESRGAN ──────────────────────────────────
train_script = '/usr/local/lib/python3.12/dist-packages/basicsr/train.py'
yaml_path    = '/content/finetune_work/aesrgan_finetune.yml'

print(f"\nUsando script : {train_script}")
print(f"Usando config : {yaml_path}")
print("Iniciando fine-tuning A-ESRGAN (batch=2, gt=128)...\n")

env = os.environ.copy()
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

result = subprocess.run(
    [sys.executable, train_script, '-opt', yaml_path],
    capture_output=True, text=True,
    cwd='/content/A-ESRGAN', env=env
)

if result.stdout:
    print("=== STDOUT ===")
    print(result.stdout[-8000:])
if result.stderr:
    print("=== STDERR ===")
    print(result.stderr[-6000:])

if result.returncode == 0:
    print("\n✅ Fine-tuning A-ESRGAN completado correctamente.")
else:
    print(f"\n[ERROR] Fine-tuning A-ESRGAN falló (returncode={result.returncode}).")


In [ ]:
# Persistir el ultimo checkpoint del fine-tuning en Google Drive.
import shutil, basicsr

# BasicSR guarda en <root>/experiments/<name>/models; <root> = carpeta de
# instalacion de basicsr (no /content). Se deriva en runtime.
basicsr_exp = Path(basicsr.__file__).resolve().parent.parent / 'experiments'

candidates = []
for root in (basicsr_exp,
             Path('/content/experiments'),
             Path('/content/finetune_work'),
             Path('/content/A-ESRGAN/experiments')):
    if root.exists():
        candidates += list(root.rglob('net_g_*.pth'))
assert candidates, 'No se encontro ningun net_g_*.pth tras el entrenamiento.'
latest = max(candidates, key=lambda p: p.stat().st_mtime)
dest = DRIVE_CKPT_DIR / latest.name
shutil.copy2(latest, dest)
print(f'Checkpoint copiado a Drive: {dest}')


## 3. Evaluación SR mínima: pre vs fine-tuned

In [ ]:
# Funcion de inferencia A-ESRGAN

def run_aesrgan_inference(
    image: Image.Image,
    model_path: Path,
    tile: int = 400,
) -> Image.Image:
    """
    Ejecuta A-ESRGAN guardando temporalmente la imagen de entrada y
    llamando al script de inferencia con el model_path indicado.
    """
    tmp_input  = WORK_DIR / 'results' / '_tmp_aesrgan_input.png'
    tmp_output = WORK_DIR / 'results' / '_tmp_aesrgan_out'
    tmp_output.mkdir(exist_ok=True)
    image.save(tmp_input)

    cmd = [
        sys.executable,
        str(AESRGAN_REPO_DIR / 'inference_aesrgan.py'),
        '--model_path', str(model_path),
        '--input',      str(tmp_input),
        '--output',     str(tmp_output),
        '--suffix',     'eval',
        '--tile',       str(tile),
    ]
    if DEVICE == 'cuda':
        cmd.append('--half')

    subprocess.run(cmd, cwd=str(AESRGAN_REPO_DIR), check=True)

    out_files = sorted(tmp_output.glob('*.png'))
    assert out_files, f'A-ESRGAN no generó salida en {tmp_output}'
    result = Image.open(out_files[-1]).convert('RGB')
    return result

In [ ]:
import numpy as np, pandas as pd
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import lpips, torch

AESRGAN_PRETRAINED = Path('/content/A-ESRGAN/experiments/pretrained_models/A_ESRGAN_Single.pth')
_ft = sorted(DRIVE_CKPT_DIR.glob('net_g_*.pth'))
AESRGAN_FT_CKPT = _ft[-1] if _ft else None
if AESRGAN_FT_CKPT is None:
    print('[WARN] Sin checkpoint fine-tuneado; solo se evalúa el pre-entrenado.')

_lpips = lpips.LPIPS(net='alex').to(DEVICE).eval()
def _lpips_dist(a, b):
    ta = torch.from_numpy(a).permute(2,0,1)[None].float().to(DEVICE)/127.5 - 1
    tb = torch.from_numpy(b).permute(2,0,1)[None].float().to(DEVICE)/127.5 - 1
    with torch.no_grad():
        return float(_lpips(ta, tb).item())

val_stems = sorted(p.stem for p in VAL_HR.glob('*.png'))
rows = []
for stem in val_stems:
    hr = np.array(Image.open(VAL_HR/f'{stem}.png').convert('RGB'))
    lr = Image.open(VAL_LR/f'{stem}.png').convert('RGB')
    pred_pre = np.array(run_aesrgan_inference(lr, AESRGAN_PRETRAINED).resize(hr.shape[1::-1]))
    row = {'stem': stem,
           'pre_psnr': psnr(hr, pred_pre, data_range=255),
           'pre_ssim': ssim(hr, pred_pre, channel_axis=2, data_range=255),
           'pre_lpips': _lpips_dist(hr, pred_pre)}
    if AESRGAN_FT_CKPT is not None:
        pred_ft = np.array(run_aesrgan_inference(lr, AESRGAN_FT_CKPT).resize(hr.shape[1::-1]))
        row.update({'ft_psnr': psnr(hr, pred_ft, data_range=255),
                    'ft_ssim': ssim(hr, pred_ft, channel_axis=2, data_range=255),
                    'ft_lpips': _lpips_dist(hr, pred_ft)})
    rows.append(row)

df = pd.DataFrame(rows)
print(df.describe())
df

In [ ]:
import matplotlib.pyplot as plt
show = val_stems[:3]
ncol = 4 if AESRGAN_FT_CKPT is not None else 3
fig, axes = plt.subplots(len(show), ncol, figsize=(4*ncol, 4*len(show)))
if len(show) == 1: axes = axes[None, :]
for r, stem in enumerate(show):
    hr = Image.open(VAL_HR/f'{stem}.png').convert('RGB')
    lr = Image.open(VAL_LR/f'{stem}.png').convert('RGB')
    cols = [(lr.resize(hr.size, Image.NEAREST), 'LR'),
            (run_aesrgan_inference(lr, AESRGAN_PRETRAINED).resize(hr.size), 'A-ESRGAN pre')]
    if AESRGAN_FT_CKPT is not None:
        cols.append((run_aesrgan_inference(lr, AESRGAN_FT_CKPT).resize(hr.size), 'A-ESRGAN fine-tuned'))
    cols.append((hr, 'HR (GT)'))
    for c,(im,t) in enumerate(cols):
        axes[r][c].imshow(im); axes[r][c].set_title(t); axes[r][c].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Cualitativa en fotos antiguas reales (sin GT): juzga el gap sintético->real.
import kagglehub
real_root = Path(kagglehub.dataset_download('marcinrutecki/old-photos'))
real_paths = []
for ext in ('*.png','*.jpg','*.jpeg'):
    real_paths.extend(sorted(real_root.rglob(ext)))
real_paths = real_paths[:3]

fig, axes = plt.subplots(len(real_paths), 2 if AESRGAN_FT_CKPT is None else 3,
                         figsize=(12, 4*len(real_paths)))
if len(real_paths) == 1: axes = axes[None, :]
for r, p in enumerate(real_paths):
    im = Image.open(p).convert('RGB')
    cols = [(im, 'Real (entrada)'),
            (run_aesrgan_inference(im, AESRGAN_PRETRAINED), 'A-ESRGAN pre')]
    if AESRGAN_FT_CKPT is not None:
        cols.append((run_aesrgan_inference(im, AESRGAN_FT_CKPT), 'A-ESRGAN fine-tuned'))
    for c,(img,t) in enumerate(cols):
        axes[r][c].imshow(img); axes[r][c].set_title(t); axes[r][c].axis('off')
plt.tight_layout(); plt.show()
print('Juzga a ojo: ¿el fine-tuned limpia mejor el grano/ruido de época sin inventar texturas?')

## Conclusiones

Enfoque rediseñado: HR "vintage nítido" (DIV2K toneado) + degradación realista
de alto orden (Real-ESRGAN al vuelo en train; `degrade_for_sr` para el val fijo).
Métricas PSNR/SSIM/**LPIPS** pre vs fine-tuned sobre el val fijo, más evaluación
cualitativa en fotos antiguas reales. El checkpoint queda en
`MyDrive/TFM/checkpoints/aesrgan_finetuned/`.